# Studio di copertura degli intervalli di credibilita' (mondi ripetuti)

**Plug and play**: carica SOLO questo notebook (File -> Carica notebook), runtime GPU
(T4), esegui le due celle in ordine. La cella di setup clona `main`, applica da sola
le patch necessarie (flag `--seed-mondo` nel generatore + **fix del controllo
stagionale**: `float_format` che impedisce a `0.9290` di diventare `929`) e scrive il
runner. Niente file da trascinare.

**Cosa misura.** La copertura e' una proprieta' su mondi ripetuti: si rigenera il
mondo, si rifitta con la STESSA config dei run canonici, si conta quante volte il vero
cade nell'intervallo al 90% e si registra il **PIT**. Otto PIT schiacciati sotto 0,05
sono evidenza schiacciante; otto PIT sparsi fra 0,1 e 0,9 dicono che e' calibrato.

**Config, identica per tutti i mondi:** 96 nodi (12/trimestre), 4 catene, adapt 500,
burnin 500, keep 2000, seed MCMC 42, prior deboli LogNormal(0.2, 0.9),
revenue_per_kpi 40.0 EUR. Costo: ~10 min a mondo, ~80 min per 8 mondi.

## Il mondo 42: prima domanda a cui risponde

Il mondo 42 e' il dataset canonico della tesi, MA generato col fix del controllo
stagionale. La sua riga risponde a: **il 2,26 si muove senza il bug?**
- se resta ~2,26 con PIT ~0,0004 -> il bug era immateriale per le stime E il
  meccanismo e' validato: lascia correre gli altri 7 mondi;
- se si muove in modo netto -> il bug era materiale: FERMA il ciclo dopo il mondo 42
  e decidiamo insieme (la questione diventa il golden run, non la copertura).

## La lettura degli 8 mondi, fissata PRIMA di eseguire

| Esito | Lettura |
|---|---|
| copertura 7-8 su 8, PIT sparsi | intervallo calibrato; il dataset canonico era un mondo sfortunato |
| copertura 4-6, PIT irregolari | copertura degradata ma non collassata; serve N piu' grande |
| copertura 0-3, PIT tutti < 0,05 | l'intervallo al 90% non vale il 90% in regime osservazionale: il modello non sa di sbagliare |

Con N = 8 l'incertezza sulla copertura resta ampia: risultato **indicativo**, e il
numero di mondi va scritto accanto alla percentuale, sempre.

**Note operative.** Ogni mondo esegue le STESSE celle (2,3,5,6,7,9) del notebook di
tesi, in un sottoprocesso che muore a fine misura (GPU liberata). Il CSV su Drive
cresce di una riga a OGNI mondo: se Colab cade al sesto, i primi cinque ci sono gia',
e rieseguire la cella riprende dai mancanti. Nessun pickle resta su disco.

In [ ]:
# ============================================================================
# CELLA A - SETUP PLUG AND PLAY: clona main, si patcha da solo, scrive il runner
# ============================================================================
import base64, json, os, shutil, subprocess, sys

REPO = '/content/mmm-bayesiano-recruiting'

_BLOB = (
    '...'
    'KVxuIn0=')

_dati = json.loads(base64.b64decode(''.join(_BLOB)).decode('utf-8'))

# --- 1. repo: clona main (una cartella parziale di un clone fallito va rimossa)
if os.path.isdir(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    print('[setup] rimuovo la cartella parziale di un clone fallito: ' + REPO)
    shutil.rmtree(REPO)
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/Giacomod2001/mmm-bayesiano-recruiting',
                    REPO], check=True)
else:
    print('[setup] repo gia' + chr(39) + ' presente, non riclonato')
print('commit: ' + subprocess.run(['git', '-C', REPO, 'log', '-1', '--oneline'],
                                  capture_output=True, text=True).stdout.strip())

# --- 2. patch del generatore (idempotente: se gia' patchato, salta) ----------
_gen = os.path.join(REPO, 'genera_dati_simulati.py')
_src = open(_gen, encoding='utf-8').read()
if '--seed-mondo' in _src and 'float_format' in _src:
    print('[setup] generatore gia' + chr(39) + ' patchato (repo aggiornato): salto')
else:
    _n = 0
    for _old, _new in _dati['patches']:
        if _new in _src:
            continue                       # questa patch c'e' gia'
        if _src.count(_old) != 1:
            raise SystemExit('PATCH NON APPLICABILE: il generatore su main e' + chr(39) +
                             ' cambiato rispetto a quello atteso. Non proseguo con un ' +
                             'file a meta' + chr(39) + ': segnalalo in chat a Claude. ' +
                             'Frammento non trovato: ' + _old[:80])
        _src = _src.replace(_old, _new)
        _n += 1
    open(_gen, 'w', encoding='utf-8').write(_src)
    print('[setup] generatore patchato (' + str(_n) + ' sostituzioni: --seed-mondo + ' +
          'fix controllo stagionale float_format)')
compile(open(_gen, encoding='utf-8').read(), _gen, 'exec')

# --- 3. runner ---------------------------------------------------------------
_run = os.path.join(REPO, 'studio_copertura_runner.py')
open(_run, 'w', encoding='utf-8').write(_dati['runner'])
compile(_dati['runner'], _run, 'exec')
print('[setup] runner scritto: ' + _run)

# autotest rapido: il flag esiste davvero
_h = subprocess.run([sys.executable, _gen, '--help'], capture_output=True, text=True)
assert '--seed-mondo' in _h.stdout, 'autotest fallito: --seed-mondo assente dall' + chr(39) + 'help'
print('[setup] autotest --seed-mondo: OK')

# --- 4. dipendenze pinnate (come la CELLA 1 del notebook di tesi) ------------
if subprocess.run([sys.executable, '-c', 'import meridian'], capture_output=True).returncode != 0:
    print('[setup] installo google-meridian==1.8.0 (qualche minuto)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'google-meridian==1.8.0', 'openpyxl', 'statsmodels'], check=True)
print('[setup] meridian pronto')

# --- 5. Drive per il CSV (unico momento interattivo: di' SI al popup) --------
CSV_RISULTATI = '/content/copertura_risultati.csv'   # fallback locale
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CSV_RISULTATI = '/content/drive/MyDrive/MMM_copertura/copertura_risultati.csv'
except Exception as e:
    print('[AVVISO] Drive non montato (' + str(e)[:80] + '): il CSV restera' + chr(39) +
          ' locale e MORIRA' + chr(39) + ' col runtime. Scaricalo a fine ciclo.')
os.makedirs(os.path.dirname(CSV_RISULTATI), exist_ok=True)
print('[setup] CSV dei risultati: ' + CSV_RISULTATI)
print()
print('SETUP COMPLETO: esegui la cella del ciclo.')


In [ ]:
# ============================================================================
# CELLA B - IL CICLO: genera -> fitta -> misura -> registra -> libera, N mondi
# ============================================================================
# LETTURA FISSATA PRIMA DI ESEGUIRE (dal mandato, non negoziata a posteriori):
#   copertura 7-8 su 8 con PIT sparsi  -> intervallo calibrato; il canonico era
#                                         un mondo sfortunato
#   copertura 4-6, PIT irregolari      -> degradata ma non collassata; serve N
#                                         piu' grande
#   copertura 0-3, PIT tutti < 0,05    -> il 90% non vale il 90% in regime
#                                         osservazionale: il modello non sa di
#                                         sbagliare
# Con N=8 il risultato e' INDICATIVO: scrivere sempre il numero di mondi
# accanto alla percentuale.
#
# MONDO 42 (il primo): dataset canonico + fix del controllo stagionale.
# La sua riga risponde a "il 2,26 si muove senza il bug?":
#   resta ~2,26 (PIT ~0,0004)  -> bug immateriale, meccanismo validato: lascia
#                                 correre gli altri 7 mondi
#   si muove in modo netto     -> bug MATERIALE: ferma il ciclo (Runtime ->
#                                 Interrompi) e portiamo la questione sul
#                                 golden run, non sulla copertura
import csv, os, subprocess, sys

SEMI_MONDO = [42, 101, 102, 103, 104, 105, 106, 107]
KEEP = 2000        # come i run canonici; se lo riduci, riducilo per TUTTI i
                   # mondi e dichiaralo: misureresti la copertura della TUA
                   # configurazione, non di quella della tesi
RUNNER = os.path.join(REPO, 'studio_copertura_runner.py')

def righe_ok(percorso):
    fatti = set()
    if os.path.exists(percorso) and os.path.getsize(percorso) > 0:
        with open(percorso, newline='', encoding='utf-8') as f:
            for r in csv.DictReader(f):
                if r.get('esito') == 'ok':
                    fatti.add(int(r['seed_mondo']))
    return fatti

gia_fatti = righe_ok(CSV_RISULTATI)
print('mondi gia' + chr(39) + ' completati nel CSV: ' + (str(sorted(gia_fatti)) if gia_fatti else 'nessuno'))

for seme in SEMI_MONDO:
    if seme in gia_fatti:
        print('mondo ' + str(seme) + ': gia' + chr(39) + ' misurato, salto (ripresa).')
        continue
    print()
    print('=' * 72)
    print('MONDO ' + str(seme) + ' - genera, fitta (~10 min), misura, registra')
    print('=' * 72)
    # sottoprocesso: stdin CHIUSO cosi' un input() imprevisto e' un errore
    # immediato, non un blocco infinito; a fine processo la GPU torna libera.
    proc = subprocess.run(
        [sys.executable, RUNNER, '--seed-mondo', str(seme),
         '--repo', REPO, '--csv', CSV_RISULTATI, '--keep', str(KEEP)],
        stdin=subprocess.DEVNULL)
    with open(CSV_RISULTATI, newline='', encoding='utf-8') as f:
        ultima = list(csv.DictReader(f))[-1]
    print('riga registrata: esito=' + ultima['esito'] +
          ' | dentro=' + str(ultima.get('dentro')) +
          ' | PIT=' + str(ultima.get('pit')) +
          ' | rapporto mediana/vero=' + str(ultima.get('rapporto_mediana')) +
          (' | ERRORE: ' + ultima['errore'][:120] if ultima.get('errore') else ''))
    if seme == 42 and ultima['esito'] == 'ok':
        print()
        print('>>> MONDO 42 (canonico + fix): rapporto mediana/vero = ' +
              str(ultima.get('rapporto_mediana')) + ' (canonico con bug: ~2,26; ' +
              'PIT canonico: ~0,0004). Se il valore si e' + chr(39) + ' mosso in modo ' +
              'netto, FERMA il ciclo e riportalo in chat.')

# --- riepilogo finale: la cosa da guardare sono i PIT in ordine --------------
print()
print('#' * 72)
print('RIEPILOGO STUDIO DI COPERTURA')
print('#' * 72)
with open(CSV_RISULTATI, newline='', encoding='utf-8') as f:
    righe = [r for r in csv.DictReader(f) if r['esito'] == 'ok'
             and int(r['seed_mondo']) in SEMI_MONDO]
n = len(righe)
dentro = sum(int(r['dentro']) for r in righe)
print('copertura: ' + str(dentro) + ' mondi su ' + str(n) +
      ' contengono il vero (intervallo al 90%; N=' + str(n) + ', indicativo)')
pits = sorted(float(r['pit']) for r in righe)
print('PIT in ordine (la cosa da guardare): ' +
      ', '.join(format(p, '.4f') for p in pits))
print('  se il modello fosse calibrato: sparsi fra 0 e 1; tutti < 0,05 = ')
print('  evidenza schiacciante di intervalli troppo stretti/spostati')
for r in sorted(righe, key=lambda x: int(x['seed_mondo'])):
    if not int(r['dentro']):
        verso = 'SOVRASTIMA (mediana sopra il vero)' \
            if float(r['rapporto_mediana']) > 1 else 'SOTTOSTIMA'
        print('mondo ' + r['seed_mondo'] + ': vero FUORI, ' + verso +
              ' - rapporto mediana/vero=' + r['rapporto_mediana'] +
              ', forbice=[' + r['rapporto_ci05'] + ', ' + r['rapporto_ci95'] +
              ']x il vero, distanza=' + str(r['sd_dalla_mediana']) + ' sd')
print('canali dentro su 7, per mondo: ' +
      ', '.join(r['seed_mondo'] + ':' + r['canali_dentro_su_7'] for r in righe))
with open(CSV_RISULTATI, newline='', encoding='utf-8') as f:
    falliti = [r for r in csv.DictReader(f) if r['esito'] != 'ok']
if falliti:
    print()
    print('ATTENZIONE: ' + str(len(falliti)) + ' run NON ok nel CSV ' +
          '(esiti: ' + ', '.join(sorted({r['esito'] for r in falliti})) + '): ' +
          'righe da guardare prima di usare il riepilogo.')
